In [11]:
import torch
import numpy as np

from dinosaw.helpers import add_custom_font, get_model, get_features
from dinosaw.linear_probe import do_linear_probe, RampTypes, LinearProbeResult, get_ramp, gen_sample_mask

from os import listdir
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

In [12]:
selected_model = 'alibi_dv2_coco'
# model = get_model(selected_model, '../../trained_models', device=DEVICE, conf_path='../../dinov3')
model = get_model(selected_model, '../../trained_models', device=DEVICE)
n_dims = 768 if '_b' in selected_model else 384
# n_dims = 256

In [13]:
ds_folder = 'data/linear_probe/homog_micros'
image_files = [f for f in listdir(ds_folder)]
n_imgs = len(image_files)

features = []
for img_file in image_files:
    img_path = f'{ds_folder}/{img_file}'
    img = Image.open(img_path).convert('RGB')
    feats = get_features(model, img, device=DEVICE, channel_last=True)
    features.append(feats)

In [14]:
ramps: tuple[RampTypes, ...] = ('lr', 'ud', 'diag', 'radial')
ramps_to_results: dict[RampTypes, list[LinearProbeResult]] = {r: [] for r in ramps}
MASK_CUTOFF_FRAC = 1
STEP = 6
RANDOM_MASK = True

for ramp in ramps:
    for i in range(n_imgs):
        feats = features[i]
        result = do_linear_probe(feats, ramp, probe_by_channel=True, mask_step=STEP, mask_cutoff_frac=MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
        ramps_to_results[ramp].append(result)

In [15]:
from skimage.transform import resize
def average_results(results: list[LinearProbeResult]) -> tuple[np.ndarray, np.ndarray, float, float, np.ndarray]:
    channel_scores_arr = np.array([res['per_channel_scores'] for res in results])
    mean_channel_scores = np.mean(channel_scores_arr, axis=0)
    std_channel_scores = np.std(channel_scores_arr, axis=0)

    scores_arr = np.array([res['stack_r_squared'] for res in results])
    mean_score = np.mean(scores_arr, axis=0)
    std_score = np.std(scores_arr, axis=0)

    n_pred_dims = results[0]['stack_pred'].shape[-1]
    mean_pred = np.zeros((34, 34, n_pred_dims))

    for res in results:
        pred = resize(res['stack_pred'], mean_pred.shape, order=1)
        mean_pred += pred / len(results)

    return mean_channel_scores, std_channel_scores, mean_score, std_score, mean_pred

In [16]:
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

def add_red_square_overlay(ax: plt.Axes, mask: np.ndarray, sf_h: float, sf_w: int) -> None:
    patches: list[Rectangle] = []
    y_inds, x_inds  = np.nonzero(mask)
    for x, y in zip(x_inds, y_inds):
        rect = Rectangle((x - sf_w / 2, y - sf_h / 2), sf_w, sf_h)
        patches.append(rect)
    pc = PatchCollection(patches, color='red', facecolor='none')
    ax.add_collection(pc)

In [17]:
%%capture
n_rows, n_cols = 4, 4
TITLE_PAD = 20
FS = 26
add_custom_font('resources/fonts', 'Grotesk')
W, H = 3, 2.5

w_spacing = [1 for _ in range(n_cols)]
SPACE_ROW_IDXS = (4,)

FIG_B_COL_OFFSET = 1
FIG_B_W_COLS = 2


fig = plt.figure(figsize=(W * n_cols, H * n_rows))
gs = GridSpec(n_rows, n_cols, figure=fig, width_ratios=w_spacing, wspace=0.25)
colors: dict[RampTypes, str] = {
    'lr': '#16ce37',
    'ud': '#87f1ab',
    'diag': '#5f978b',
    'radial': '#133b32',
}
ramp_to_title: dict[RampTypes, str] = {
    'lr': 'Left-right',
    'ud': 'Up-down',
    'diag': 'Diagonal',
    'radial': 'Radial',
}
# spacer_row = fig.add_subplot(gs[:, 5])


top_left_ramp_ax = None
for row, ramp in enumerate(ramps):
    h, w = 34, 34
    ramp_arr = get_ramp(ramp, h, w)
    ramp_ax = fig.add_subplot(gs[row, 0])
    ramp_ax.imshow(ramp_arr, cmap='viridis', vmin=0, vmax=1)

    mask = gen_sample_mask((h, w), ramp, STEP, MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
    add_red_square_overlay(ramp_ax, mask, 1, 1)

    ramp_ax.set_xticks([])
    ramp_ax.set_yticks([])

    ramp_ax.set_ylabel(ramp_to_title[ramp], fontsize=FS)

    if row == 0:
        ramp_ax.set_title('Target ramp', fontsize=FS, pad=TITLE_PAD)
        top_left_ramp_ax = ramp_ax


worst_channels = []
for row, ramp in enumerate(ramps):
    ax = fig.add_subplot(gs[row, FIG_B_COL_OFFSET:FIG_B_COL_OFFSET + FIG_B_W_COLS])
    mean_channel_scores, std_channel_scores, mean_score, _, mean_pred = average_results(ramps_to_results[ramp])

    print(f"{ramp}: {np.argsort(-mean_channel_scores)[:3]}")
    worst_channels.extend(list(np.argsort(-mean_channel_scores)[:2]))

    ax.hlines(0, 0, n_dims, 'red', '--')

    ax.plot(mean_channel_scores, color=colors[ramp])
    ax.fill_between(
        np.arange(n_dims),
        mean_channel_scores - std_channel_scores,
        mean_channel_scores + std_channel_scores,
        color=colors[ramp],
        alpha=0.3,
    )
    ax.set_ylim(-0.1, 1)
    ax.set_xlim(0 -10, n_dims + 10)
    ax.tick_params(axis='both', labelsize=FS - 6)

    # ax.ticklabel_format(size=FS - 4)

    mean_pred_ax = fig.add_subplot(gs[row, FIG_B_COL_OFFSET + FIG_B_W_COLS])
    mean_pred_ax.imshow(mean_pred, cmap='viridis', vmin=0, vmax=1)

    if row == 0:
        ax.set_title('Per-channel ' + r'$R^2$' +  'scores', fontsize=FS, pad=TITLE_PAD)
        mean_pred_ax.set_title('Mean prediction \n(all channels)', fontsize=FS)
    elif row == len(ramps) - 1:
        ax.set_xlabel('Channel', fontsize=FS)
    

    mean_pred_ax.set_ylabel(f'$R^{2}:${mean_score:.2f}', fontsize=FS)
    mean_pred_ax.set_xticks([])
    mean_pred_ax.set_yticks([])



plt.tight_layout(pad=0.05)
# plt.savefig('saved/02.png', dpi=300, bbox_inches='tight')
plt.savefig("saved/09.jpeg", dpi=300, bbox_inches='tight', pil_kwargs={'optimize': True})